# KMA

In [3]:
import os
import json
cwd = os.getcwd()

In [6]:
with open(f"{cwd}/data/sajin_260409.jsonl", "r", encoding="utf-8") as f:
    s_json_list = list(f)

s_set= []
for json_str in s_json_list:
    result = json.loads(json_str)
    s_set.append(result)

###

with open(f"{cwd}/data/marcel_260409.jsonl", "r", encoding="utf-8") as f:
    m_json_list = list(f)

m_set= []
for json_str in m_json_list:
    result = json.loads(json_str)
    m_set.append(result)

In [39]:
s_set

[{'id': 31123,
  'text': 'Type of modification 23 General information on the request for amendment 23 Type of amendment 23 Detailed information on the specific elements of each modification 24 Amendment to AECM General and Training to Implement Agri-Environment-Climate Measure.24 Reasonsthat justify the change 24 Expected effects of the change 24 The impact of the change on targets and indicators 24 The impact of the change on the financing plan. 24 Amendment to GAEC 2 .24 Reasonsthat justify the change 24 Expected effects of the change 25 The impact of the change on targets and indicators 25 The impact of the change on the financing plan. 25 Changes to Annex 7.3 EAFRD.. 26 Reasonsthat justify the change 26 Expected effects of the change 26 The impact of the change on targets and indicators 26 The impact of the change on the financing plan. 26 Changes to Annex 7.3 EAGF 26 Reasonsthat justify the change 26 Expected effects of the change 26 The impact of the change on targets and indicat

In [20]:
s_ids = [se['chunk_id'] for se in s_set]
m_ids = [me['chunk_id'] for me in m_set]
shared_ids = list(set(s_ids) & set(m_ids))
len(s_ids), len(m_ids), len(shared_ids)

(501, 699, 176)

## Sentence-Based

In [21]:
comparisons = {
    "sajin":{
        "none": [],
        "smth": []
    },
    "marcel":{
        "none": [],
        "smth": []
    }
}

for se in s_set:
    if se['chunk_id'] in shared_ids:
        if len(se['label']) == 0:
            comparisons['sajin']['none'].append(se['chunk_id'])
        else:
            comparisons['sajin']['smth'].append(se['chunk_id'])
for me in m_set:
    if me['chunk_id'] in shared_ids:
        if len(me['label']) == 0:
            comparisons['marcel']['none'].append(me['chunk_id'])
        else:
            comparisons['marcel']['smth'].append(me['chunk_id'])

In [24]:
# before filtering for shared annotations:
# s: 262 239
# m: 630 69
# after
# s: 73 109
# m: 157 19
print(len(comparisons['sajin']['none']), len(comparisons['sajin']['smth']))
print(len(comparisons['marcel']['none']), len(comparisons['marcel']['smth']))

73 103
157 19


In [32]:
from datasets import Dataset, Features, Value, Sequence

features = Features({
    'id': Value('int64'), 
    'text': Value('string'),
    'pg': Value('string'),
    'chunk_id': Value('string'),
    'label': Sequence(Sequence(Value('string'))),
    'Comments': Sequence(Value('string'))
})

s_ds = Dataset.from_list(s_set, features=features)
m_ds = Dataset.from_list(m_set, features=features)

In [34]:
checking = {idx:{} for idx in comparisons['marcel']['smth']}

for se in s_set:
    if se['chunk_id'] in comparisons['marcel']['smth']:
        checking[se['chunk_id']]['text'] = se['text']
        checking[se['chunk_id']]['s_annot'] = se['label']
for me in m_set:
    if me['chunk_id'] in comparisons['marcel']['smth']:
        checking[me['chunk_id']]['m_annot'] = me['label']

In [38]:
for idx in list(checking):
    print("\n", idx)
    print("text:", checking[idx]['text'])
    s_an = checking[idx]["s_annot"]
    m_an = checking[idx]["m_annot"]
    s_txts = [(checking[idx]['text'][slbl[0]:slbl[1]], slbl[2]) for slbl in s_an]
    m_txts = [(checking[idx]['text'][mlbl[0]:mlbl[1]], mlbl[2]) for mlbl in m_an]
    print("s_annot:", s_txts)
    print("m_annot:", m_txts)
    


 319_0_19
text: (1) As regards the mapping aspect (point (i) above), while it was initially planned that the work on identification and mapping of peatland/wetland areas would be completed by the end of 2023, this is no longer feasible, despite ongoing efforts to collect all necessary data and evidence on time including research outputs. During the course of 2022/2023, consideration was given to using existing mapping data for the implementation of GAEC 2. However, DAFM found that technical issues related to the scale of the current map (i.e. the peatland layers vary in scale from \(1:100,000 - 1:150,000\) compared to \(1:5,000\) for the LPIS layer), and some mismatches between the peatland map layer boundaries and the LPIS parcel boundaries, poses some challenges when intersecting these layers on the LPIS and defining the areas to be covered by GAEC 2. Moreover, upcoming results of ongoing research projects, aimed at mapping and monitoring peatlands/wetland, are expected to add essen

## Token-Based

###